# 🤖 Dynamic Website Scraping with Selenium
### PixelCraft Infotech — Classroom Teaching Notebook

---

| Topic | Details |
|---|---|
| **Tool** | `Selenium` + `ChromeDriver` |
| **Also Covers** | `Playwright`, `Scrapy`, `API scraping` |
| **Lessons** | 12 lessons with live exercises |
| **Practice Sites** | quotes.toscrape.com/js · books.toscrape.com |

---

## ❓ Static vs Dynamic — What's the Difference?

| | Static Website | Dynamic Website |
|---|---|---|
| **HTML** | Sent directly from server | Built by JavaScript in browser |
| **Tool** | `requests` + `BeautifulSoup` | `Selenium` / `Playwright` |
| **Example** | books.toscrape.com | Twitter, Amazon, Instagram |
| **Speed** | ⚡ Very fast | 🐢 Slower (opens real browser) |

> ⚠️ If `requests.get()` returns empty content or missing data → the site is **dynamic** → use **Selenium**!

---

## 📚 Table of Contents
1. [Why Selenium? The Problem with Dynamic Sites](#lesson1)
2. [Installing Selenium & ChromeDriver](#lesson2)
3. [Launching the Browser](#lesson3)
4. [Finding Elements with Selenium](#lesson4)
5. [Clicking Buttons & Interactions](#lesson5)
6. [Filling Forms & Searching](#lesson6)
7. [Waiting for Elements (Explicit Wait)](#lesson7)
8. [Scrolling & Infinite Scroll](#lesson8)
9. [Taking Screenshots](#lesson9)
10. [Headless Mode (No Browser Window)](#lesson10)
11. [Scraping APIs (The Smart Way)](#lesson11)
12. [Mini Project: Dynamic Quotes Scraper](#lesson12)

---
## 🔵 LESSON 1 — Why Selenium? The Problem with Dynamic Sites

### The Problem:
Many modern websites **load content using JavaScript** after the page opens.

When you use `requests.get()`, Python gets the HTML **before JavaScript runs** — so the data isn't there yet!

```
Browser Flow:   URL → HTML arrives → JavaScript runs → Content appears ✅
requests Flow:  URL → HTML arrives → STOP ❌ (JavaScript never runs)
```

### The Solution:
**Selenium** opens a **real browser** and waits for JavaScript to finish — then scrapes the final content.

### When to use what:
| Situation | Tool |
|---|---|
| Content visible in page source | `requests` + `BeautifulSoup` |
| Content loaded by JavaScript | `Selenium` |
| Need to click/scroll/login | `Selenium` |
| Website has a public API | Scrape the **API directly** (fastest!) |

In [ ]:
# ── DEMO: The Problem with Dynamic Sites ─────────────
import requests
from bs4 import BeautifulSoup

# This version of the site loads quotes using JavaScript
url = "https://quotes.toscrape.com/js/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "lxml")

quotes = soup.find_all("div", class_="quote")
print(f"Quotes found with requests: {len(quotes)}")
print()

# ❌ Result: 0 quotes found!
# Because JavaScript hasn't run — the content isn't in the HTML yet
print("❌ Zero quotes found — JavaScript hasn't run!")
print("   We need Selenium to open a real browser.")

---
## 🔵 LESSON 2 — Installing Selenium & ChromeDriver

### Step 1: Install the Selenium library
```bash
pip install selenium
pip install webdriver-manager
```

### Step 2: ChromeDriver
Selenium needs a **driver** to control Chrome browser.

- `webdriver-manager` **automatically** downloads the correct driver for you!
- No manual download needed ✅

### Tool Comparison:
| Tool | Speed | Difficulty | Best For |
|---|---|---|---|
| `Selenium` | Medium | ⭐ Easy | Most dynamic sites |
| `Playwright` | Fast | ⭐⭐ Medium | Modern apps, async |
| `Scrapy` | Very Fast | ⭐⭐⭐ Hard | Large-scale scraping |

In [ ]:
# ── INSTALL SELENIUM ──────────────────────────────────
import sys
!{sys.executable} -m pip install selenium webdriver-manager --quiet
print("✅ Selenium installed!")

In [ ]:
# ── IMPORT SELENIUM TOOLS ────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

import time
import json
import csv

print("✅ All Selenium imports successful!")

---
## 🔵 LESSON 3 — Launching the Browser

Selenium can open a **real Chrome window** that you can see, or run **invisibly** (headless mode).

```python
driver = webdriver.Chrome()    # Opens a visible Chrome window
driver.get(url)                # Navigate to a URL
driver.quit()                  # Always close when done!
```

> 💡 **Always call `driver.quit()`** at the end — otherwise Chrome windows keep running in the background!

In [ ]:
# ── HELPER: Create driver with common options ─────────
def create_driver(headless=False):
    """
    Create and return a Chrome WebDriver.
    headless=True  → No browser window (runs in background)
    headless=False → Opens a visible Chrome window
    """
    options = Options()

    if headless:
        options.add_argument("--headless")         # No window

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")  # Hide bot fingerprint
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/120.0.0.0 Safari/537.36")

    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=options)
    return driver

print("✅ create_driver() helper is ready!")

In [ ]:
# ── OPEN A WEBSITE ───────────────────────────────────
driver = create_driver(headless=True)  # headless=True for classroom demo

driver.get("https://quotes.toscrape.com/js/")

print("Page Title  :", driver.title)
print("Current URL :", driver.current_url)
print("Window Size :", driver.get_window_size())

# Always close the driver when done
driver.quit()
print("\n✅ Browser closed cleanly.")

---
## 🔵 LESSON 4 — Finding Elements with Selenium

Selenium uses `By` class to locate elements — similar to BeautifulSoup but more powerful.

| Method | Code | Like BeautifulSoup |
|---|---|---|
| By CSS class | `By.CLASS_NAME` | `class_="name"` |
| By CSS selector | `By.CSS_SELECTOR` | `soup.select()` |
| By tag/XPath | `By.XPATH` | No equivalent |
| By ID | `By.ID` | `id="name"` |
| By link text | `By.LINK_TEXT` | — |

```python
driver.find_element(By.CSS_SELECTOR, ".quote")     # First match
driver.find_elements(By.CSS_SELECTOR, ".quote")    # All matches (list)
```

In [ ]:
# ── FINDING ELEMENTS ─────────────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")

time.sleep(2)                          # Wait for JS to load

# ── find_element → returns ONE element ───────────────
first_quote = driver.find_element(By.CLASS_NAME, "quote")
print("First quote text:")
print(first_quote.text[:100])

# ── find_elements → returns a LIST ───────────────────
all_quotes = driver.find_elements(By.CLASS_NAME, "quote")
print(f"\nTotal quotes found: {len(all_quotes)}")

# ── CSS Selector (most flexible) ─────────────────────
authors = driver.find_elements(By.CSS_SELECTOR, ".quote .author")
print("\nAll authors:")
for author in authors:
    print(" ", author.text)

driver.quit()

In [ ]:
# ── ELEMENT METHODS ──────────────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

quote_el = driver.find_element(By.CLASS_NAME, "quote")

print("element.text          :", quote_el.text[:60])

# Get inner HTML
inner_html = quote_el.get_attribute("innerHTML")
print("\nInner HTML (first 200):", inner_html[:200])

# Get a specific attribute
link_el = driver.find_element(By.CSS_SELECTOR, ".quote a")
print("\nLink href:", link_el.get_attribute("href"))

# Check if element is visible
print("Is displayed:", quote_el.is_displayed())

driver.quit()

In [ ]:
# 🔴 CLASSROOM EXERCISE 1
# Find all TAG elements on the quotes page
# Print the unique tags found

driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

# YOUR CODE: Find all elements with class 'tag'
tags = driver.find_elements(By.CLASS_NAME, "tag")

print(f"Total tags found: {len(tags)}")
print("Unique tags:")
unique_tags = list(set([t.text for t in tags]))
for tag in sorted(unique_tags):
    print(" ", tag)

driver.quit()

---
## 🔵 LESSON 5 — Clicking Buttons & Interactions

This is where Selenium becomes **powerful** — you can interact with pages just like a real user!

```python
element.click()                    # Click a button or link
element.send_keys("hello")         # Type into an input field
driver.back()                      # Go back like browser Back button
driver.refresh()                   # Refresh the page
```

In [ ]:
# ── CLICKING NEXT PAGE ───────────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

print("Page 1 URL:", driver.current_url)

# Count quotes on page 1
page1_quotes = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Quotes on page 1: {len(page1_quotes)}")

# Find and click the 'Next' button
next_button = driver.find_element(By.CSS_SELECTOR, "li.next a")
print("\nClicking Next button...")
next_button.click()

time.sleep(2)                          # Wait for page 2 to load

print("Page 2 URL:", driver.current_url)

# Count quotes on page 2
page2_quotes = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Quotes on page 2: {len(page2_quotes)}")

driver.quit()

In [ ]:
# ── CLICK THROUGH ALL PAGES ──────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

all_quotes_text = []
page = 1

while True:
    print(f"\n📄 Scraping page {page}...")

    quotes_els = driver.find_elements(By.CLASS_NAME, "quote")
    for q in quotes_els:
        text   = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        all_quotes_text.append({"quote": text, "author": author, "page": page})

    print(f"   ✅ {len(quotes_els)} quotes scraped")

    # Try to find the Next button
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "li.next a")
        next_btn.click()
        time.sleep(2)
        page += 1
    except:
        print("   ⏹️  No more pages — done!")
        break

driver.quit()
print(f"\n✅ Total quotes scraped: {len(all_quotes_text)}")

---
## 🔵 LESSON 6 — Filling Forms & Searching

Selenium can type into forms, fill login pages, and submit searches — exactly like a human!

```python
input_box = driver.find_element(By.NAME, "q")
input_box.clear()                      # Clear any existing text
input_box.send_keys("Python")          # Type into the box
input_box.send_keys(Keys.RETURN)       # Press Enter
```

In [ ]:
# ── FILL A LOGIN FORM ────────────────────────────────
# quotes.toscrape.com has a login page we can practice on

driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/login")
time.sleep(1)

print("Page:", driver.title)

# Find the username input
username_input = driver.find_element(By.ID, "username")
username_input.clear()
username_input.send_keys("admin")      # Type username

# Find the password input
password_input = driver.find_element(By.ID, "password")
password_input.clear()
password_input.send_keys("admin")      # Type password

print("Filled in username: admin")
print("Filled in password: admin")

# Submit the form
submit_btn = driver.find_element(By.CSS_SELECTOR, "input[type='submit']")
submit_btn.click()

time.sleep(1)
print("\nAfter login URL:", driver.current_url)
print("Logged in!", "Logout" in driver.page_source)

driver.quit()

In [ ]:
# ── SEARCH + SCRAPE RESULTS ──────────────────────────
# Searching on quotes.toscrape.com by tag

driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/tag/love/")
time.sleep(2)

print("Quotes tagged 'love':")
quote_els = driver.find_elements(By.CLASS_NAME, "quote")

for q in quote_els:
    text   = q.find_element(By.CLASS_NAME, "text").text
    author = q.find_element(By.CLASS_NAME, "author").text
    print(f"\n  {text[:70]}...")
    print(f"  — {author}")

driver.quit()

In [ ]:
# 🔴 CLASSROOM EXERCISE 2
# 1. Open quotes.toscrape.com/login
# 2. Login with username='user' and password='user'
# 3. After login, scrape the first 3 quotes
# 4. Print each quote and its author

driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/login")
time.sleep(1)

# Step 1: Fill the form
driver.find_element(By.ID, "username").send_keys("user")
driver.find_element(By.ID, "password").send_keys("user")
driver.find_element(By.CSS_SELECTOR, "input[type='submit']").click()
time.sleep(2)

# Step 2: Scrape quotes after login
quotes = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Quotes found after login: {len(quotes)}")
for q in quotes[:3]:
    print("\n  ", q.find_element(By.CLASS_NAME, "text").text[:60])
    print("   —", q.find_element(By.CLASS_NAME, "author").text)

driver.quit()

---
## 🔵 LESSON 7 — Waiting for Elements (Explicit Wait)

### The Problem:
JavaScript content takes **time to load**. If we look for elements too early — they're not there yet!

### ❌ Bad: time.sleep() — Guessing how long to wait
```python
time.sleep(5)   # ← Wastes time if page loads in 1s
                # ← Still fails if page needs 6s!
```

### ✅ Good: WebDriverWait — Smart waiting
```python
wait = WebDriverWait(driver, timeout=10)
element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))
# Waits UP TO 10 seconds, stops as soon as element appears
```

In [ ]:
# ── EXPLICIT WAIT — The RIGHT way to wait ────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")

# Create a Wait object (waits up to 10 seconds)
wait = WebDriverWait(driver, timeout=10)

# Wait until quote elements appear on the page
print("Waiting for quotes to load...")
first_quote = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, "quote"))
)
print("✅ Quotes loaded!")
print("First quote:", first_quote.text[:80])

driver.quit()

In [ ]:
# ── COMMON WAIT CONDITIONS ───────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
wait = WebDriverWait(driver, timeout=10)

# CONDITION 1: Wait until element EXISTS in DOM
el1 = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))
print("✅ EC.presence_of_element_located — element exists")

# CONDITION 2: Wait until element is VISIBLE
el2 = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "quote")))
print("✅ EC.visibility_of_element_located — element visible")

# CONDITION 3: Wait until element is CLICKABLE
el3 = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
print("✅ EC.element_to_be_clickable — element ready to click")

# CONDITION 4: Wait until ALL elements are present
all_q = wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
print(f"✅ EC.presence_of_all_elements_located — {len(all_q)} elements found")

driver.quit()

---
## 🔵 LESSON 8 — Scrolling & Infinite Scroll

Many modern websites (Twitter, Instagram, LinkedIn) use **infinite scroll** — new content loads as you scroll down.

Selenium can scroll using **JavaScript execution**:
```python
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
#                      ↑ Scroll to the very bottom of the page
```

In [ ]:
# ── SCROLLING WITH JAVASCRIPT ────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/scroll")
time.sleep(2)

print("Starting infinite scroll demo...")
all_quotes = []

for scroll_num in range(1, 6):         # Scroll 5 times

    # Get current page height before scroll
    before_height = driver.execute_script("return document.body.scrollHeight")

    # Scroll to the bottom
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")

    time.sleep(1.5)                    # Wait for new content to load

    # Collect all quotes now visible
    quotes_on_page = driver.find_elements(By.CLASS_NAME, "quote")

    print(f"  Scroll {scroll_num}: {len(quotes_on_page)} quotes visible")

    # Check if new content was loaded
    after_height = driver.execute_script("return document.body.scrollHeight")
    if after_height == before_height:
        print("  ⏹️  Page height didn't change — reached the end!")
        break

# Collect all final quotes
for q in driver.find_elements(By.CLASS_NAME, "quote"):
    text   = q.find_element(By.CLASS_NAME, "text").text
    author = q.find_element(By.CLASS_NAME, "author").text
    all_quotes.append({"text": text, "author": author})

driver.quit()
print(f"\n✅ Total quotes collected: {len(all_quotes)}")

In [ ]:
# ── DIFFERENT SCROLL TYPES ───────────────────────────
driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com")
time.sleep(1)

# Scroll to bottom of page
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
print("✅ Scrolled to BOTTOM")

time.sleep(0.5)

# Scroll back to top
driver.execute_script("window.scrollTo(0, 0)")
print("✅ Scrolled to TOP")

time.sleep(0.5)

# Scroll by a fixed amount (e.g., 500px down)
driver.execute_script("window.scrollBy(0, 500)")
print("✅ Scrolled DOWN by 500px")

time.sleep(0.5)

# Scroll a specific element into view
footer = driver.find_element(By.TAG_NAME, "footer")
driver.execute_script("arguments[0].scrollIntoView();", footer)
print("✅ Scrolled footer INTO VIEW")

driver.quit()

---
## 🔵 LESSON 9 — Taking Screenshots

Selenium can take a screenshot of whatever the browser sees — great for debugging!

```python
driver.save_screenshot("page.png")              # Full page screenshot
element.screenshot("element.png")               # Screenshot of one element
```

In [ ]:
# ── TAKE SCREENSHOTS ─────────────────────────────────
from IPython.display import Image, display

driver = create_driver(headless=True)
driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)

# Full page screenshot
driver.save_screenshot("full_page.png")
print("✅ Full page screenshot saved: full_page.png")

# Screenshot of a specific element
quote_el = driver.find_element(By.CLASS_NAME, "quote")
quote_el.screenshot("first_quote.png")
print("✅ Element screenshot saved: first_quote.png")

driver.quit()

# Display in notebook
print("\n📸 Full page screenshot:")
display(Image("full_page.png", width=700))

---
## 🔵 LESSON 10 — Headless Mode (No Browser Window)

**Headless mode** runs Chrome **invisibly** in the background — no window appears.

Use this when:
- Running on a **server** (no display available)
- Running **automated scripts** overnight
- You don't need to see the browser

```python
options.add_argument("--headless")     # ← This single line makes it headless
```

| Mode | When to Use |
|---|---|
| `headless=False` | Learning, debugging, checking what's happening |
| `headless=True` | Production scripts, servers, automation |

In [ ]:
# ── HEADLESS MODE DEMO ───────────────────────────────
import time as t

# ── Visible mode ─────────────────────────────────────
start = t.time()
driver_visible = create_driver(headless=False)
driver_visible.get("https://quotes.toscrape.com/js/")
time.sleep(2)
count_visible = len(driver_visible.find_elements(By.CLASS_NAME, "quote"))
driver_visible.quit()
time_visible = t.time() - start

# ── Headless mode ─────────────────────────────────────
start = t.time()
driver_headless = create_driver(headless=True)
driver_headless.get("https://quotes.toscrape.com/js/")
time.sleep(2)
count_headless = len(driver_headless.find_elements(By.CLASS_NAME, "quote"))
driver_headless.quit()
time_headless = t.time() - start

print("Results comparison:")
print(f"  Visible  mode → {count_visible} quotes  | Time: {time_visible:.1f}s")
print(f"  Headless mode → {count_headless} quotes  | Time: {time_headless:.1f}s")
print("\n✅ Same results — headless is faster & uses less memory!")

---
## 🔵 LESSON 11 — Scraping APIs (The Smart Way 🎯)

### The Secret Technique!

Many dynamic websites don't build HTML from scratch — they call a **hidden API** to get JSON data!

```
Browser  →  API Request  →  JSON Data  →  JavaScript builds HTML
```

If you call that API directly → you get **clean JSON** without needing Selenium at all! ⚡

### How to Find Hidden APIs:
1. Open Chrome → **F12** → **Network tab**
2. Reload the page
3. Click **Fetch/XHR** filter
4. Look for requests returning JSON
5. Copy that URL and call it with `requests`!

In [ ]:
# ── API SCRAPING EXAMPLE ─────────────────────────────
# Many sites expose a JSON API — much faster than Selenium!

import requests

# Example: JSONPlaceholder (a free public API for practice)
api_url = "https://jsonplaceholder.typicode.com/posts"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
}

response = requests.get(api_url, headers=headers)
data = response.json()             # Directly parse JSON!

print(f"✅ Got {len(data)} posts from the API!")
print("\nFirst 3 posts:")
for post in data[:3]:
    print(f"\n  ID    : {post['id']}")
    print(f"  Title : {post['title']}")
    print(f"  Body  : {post['body'][:60]}...")

In [ ]:
# ── API WITH QUERY PARAMETERS ────────────────────────
# APIs often accept parameters to filter/search data

# Get posts by a specific user
api_url = "https://jsonplaceholder.typicode.com/posts"
params  = {"userId": 1}            # Filter by userId=1

response = requests.get(api_url, params=params)
posts = response.json()

print(f"Posts by user 1: {len(posts)}")
for post in posts[:3]:
    print(f"  [{post['id']}] {post['title'][:50]}")

print("\n📌 TIP: Check the Network tab in browser DevTools")
print("   to find hidden API endpoints on any website!")

In [ ]:
# ── SAVE API DATA TO JSON ────────────────────────────
with open("api_posts.json", "w") as f:
    json.dump(posts, f, indent=4)

print("✅ Saved API data to api_posts.json")

# ── SAVE TO CSV ──────────────────────────────────────
with open("api_posts.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["userId", "id", "title", "body"])
    writer.writeheader()
    writer.writerows(posts)

print("✅ Saved API data to api_posts.csv")

---
## 🔵 LESSON 12 — Mini Project: Dynamic Quotes Scraper 🎓

**Target:** `https://quotes.toscrape.com/js/` (JavaScript-rendered!)

**Goal:** Use Selenium to:
1. Open the page and wait for JS to render
2. Scrape all quotes + authors + tags
3. Click through all pages
4. Save results to JSON and CSV

> 🎯 **Build this together in class — step by step!**

In [ ]:
# ── FULL MINI PROJECT ─────────────────────────────────
print("="*55)
print("  MINI PROJECT: Dynamic Quotes Scraper")
print("="*55)

driver     = create_driver(headless=True)
wait       = WebDriverWait(driver, timeout=10)
all_data   = []
page_count = 0

driver.get("https://quotes.toscrape.com/js/")

while True:
    page_count += 1
    print(f"\n📄 Scraping page {page_count}...")

    # Wait for quotes to load (smart wait)
    try:
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "quote")))
    except:
        print("   ⚠️  Timeout — no quotes found.")
        break

    # Scrape all quotes on this page
    quote_els = driver.find_elements(By.CLASS_NAME, "quote")

    for q in quote_els:
        text   = q.find_element(By.CLASS_NAME, "text").text.strip()
        author = q.find_element(By.CLASS_NAME, "author").text.strip()
        tags   = [t.text for t in q.find_elements(By.CLASS_NAME, "tag")]

        all_data.append({
            "quote":  text,
            "author": author,
            "tags":   ", ".join(tags),
            "page":   page_count,
        })

    print(f"   ✅ {len(quote_els)} quotes scraped")

    # Go to next page
    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a")))
        next_btn.click()
        time.sleep(1.5)
    except:
        print("   ⏹️  Last page reached.")
        break

driver.quit()

# ── SAVE RESULTS ─────────────────────────────────────
with open("dynamic_quotes.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, indent=4, ensure_ascii=False)

with open("dynamic_quotes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["quote", "author", "tags", "page"])
    writer.writeheader()
    writer.writerows(all_data)

print(f"\n{'='*55}")
print(f"  ✅ Total quotes scraped : {len(all_data)}")
print(f"  ✅ Pages scraped        : {page_count}")
print(f"  ✅ Saved to             : dynamic_quotes.json")
print(f"  ✅ Saved to             : dynamic_quotes.csv")
print(f"{'='*55}")

In [ ]:
# ── VERIFY RESULTS ────────────────────────────────────
with open("dynamic_quotes.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(f"Total records in JSON: {len(loaded)}")
print("\nSample records:")
for item in loaded[:3]:
    print(f"\n  Quote  : {item['quote'][:60]}...")
    print(f"  Author : {item['author']}")
    print(f"  Tags   : {item['tags']}")
    print(f"  Page   : {item['page']}")

In [ ]:
# 🏠 HOMEWORK CHALLENGES

# CHALLENGE 1 — Screenshot every page
# Take a screenshot of each page before clicking Next
# Save as: page_1.png, page_2.png, etc.

# CHALLENGE 2 — Author detail pages
# After scraping quotes, click on each author's name
# Scrape their born date and description
# Save as: authors.json

# CHALLENGE 3 — Login then scrape
# 1. Login with username='admin', password='admin'
# 2. Scrape all quotes after login
# 3. Compare — are there different quotes when logged in?

# CHALLENGE 4 — Tag analysis
# From all scraped quotes, find:
#   → Which author has the most quotes?
#   → What are the top 5 most used tags?
#   → Which quote has the most tags?

print("💡 Pick a challenge and start coding!")

---
## 📋 Selenium Quick Reference Cheat Sheet

### Setup
| Code | What it does |
|---|---|
| `webdriver.Chrome(service=..., options=...)` | Launch Chrome browser |
| `options.add_argument("--headless")` | Run without visible window |
| `driver.get(url)` | Navigate to a URL |
| `driver.quit()` | Close browser — always do this! |

### Finding Elements
| Code | What it does |
|---|---|
| `driver.find_element(By.CLASS_NAME, "x")` | Find first element by class |
| `driver.find_elements(By.CLASS_NAME, "x")` | Find all elements (list) |
| `driver.find_element(By.ID, "x")` | Find by HTML id |
| `driver.find_element(By.CSS_SELECTOR, "div.card h2")` | CSS selector |
| `driver.find_element(By.XPATH, "//div[@class='x']")` | XPath selector |

### Interactions
| Code | What it does |
|---|---|
| `element.click()` | Click a button or link |
| `element.send_keys("text")` | Type into an input |
| `element.send_keys(Keys.RETURN)` | Press Enter key |
| `element.clear()` | Clear an input field |
| `element.text` | Get visible text |
| `element.get_attribute("href")` | Get an attribute |

### Waiting
| Code | What it does |
|---|---|
| `WebDriverWait(driver, 10)` | Create wait (max 10 seconds) |
| `wait.until(EC.presence_of_element_located(...))` | Wait until element exists |
| `wait.until(EC.element_to_be_clickable(...))` | Wait until element clickable |
| `wait.until(EC.visibility_of_element_located(...))` | Wait until element visible |

### JavaScript
| Code | What it does |
|---|---|
| `driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")` | Scroll to bottom |
| `driver.execute_script("window.scrollTo(0, 0)")` | Scroll to top |
| `driver.execute_script("arguments[0].click();", element)` | Force click via JS |

### Other
| Code | What it does |
|---|---|
| `driver.save_screenshot("file.png")` | Take full page screenshot |
| `driver.title` | Get page title |
| `driver.current_url` | Get current URL |
| `driver.back()` | Go back |
| `driver.refresh()` | Refresh page |

---

## 🎓 End of Class — Well Done Everyone!

**Next Steps:**
- ⚡ Learn **Playwright** — faster, modern alternative to Selenium
- 🕷️ Learn **Scrapy** — for large-scale production scraping
- 🌐 Explore **browser DevTools Network tab** — find hidden APIs everywhere!
- 📊 Combine with **Pandas** — clean and analyze your scraped data
- 🗄️ Store scraped data in **SQLite / PostgreSQL**

> Made with ❤️ by **PixelCraft Infotech**